# 第 14 章 AD-AS 分析と主要理論: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/economics-programming/14-ad-as-and-major-theories.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
government = 100               # IS-LM の政府支出 G
money_supply = 525             # IS-LM の名目貨幣供給 M
government_after = 100         # 図: 政策後の政府支出
full_employment_income = 1250  # 完全雇用所得（古典派の総供給は、ここで垂直）
natural_rate = 4               # フィリップス曲線: 自然失業率（%）
expected_inflation = 2         # 期待物価上昇率（%）
phillips_slope = 0.5           # フィリップス曲線の傾き
unemployment = 2               # 失業率（%）
actual_gdp = 980               # 需給ギャップ: 実際の GDP
potential_gdp = 1000           # 潜在 GDP
nominal_rate = 5               # フィッシャー方程式: 名目利子率（%）
inflation = 2                  # インフレ率（%）
nominal_money = 500            # 貨幣数量説: 貨幣供給 M
velocity = 2                   # 貨幣の流通速度 V
real_income = 1000             # 実質所得 Y

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) in sys.path:
    sys.path.remove(str(_root))
sys.path.insert(0, str(_root))
_lib = sys.modules.get("lib")
if _lib is not None and Path(getattr(_lib, "__file__", "") or "").resolve().parent != _root / "lib":
    for _name in list(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## AD-AS 分析

In [ ]:
import dataclasses

from lib.chapter13.islm import IsLmModel
from lib.chapter14.adas import classical_supply, equilibrium_price_and_income

model = IsLmModel(government=government, money_supply=money_supply)
after = dataclasses.replace(model, government=government_after)
price, income = equilibrium_price_and_income(
    model, lambda p: classical_supply(p, full_employment_income)
)
price_after, income_after = equilibrium_price_and_income(
    after, lambda p: classical_supply(p, full_employment_income)
)
print(f"均衡: 物価 {price:.3f}、所得 {income:.1f}")
print(f"政策後: 物価 {price_after:.3f}、所得 {income_after:.1f}")

In [ ]:
from lib.chapter14.plot import plot_adas

plot_adas(model, full_employment_income, after)

## フィリップス曲線・需給ギャップ・オークンの法則・その他

In [ ]:
from lib.chapter14.indicators import (
    okun_unemployment,
    output_gap,
    phillips_inflation,
    price_level_by_quantity_theory,
    real_interest_rate,
)

inflation_rate = phillips_inflation(
    unemployment, natural_rate, expected_inflation, phillips_slope
)
print(f"物価上昇率（フィリップス曲線）: {inflation_rate:.2f}%")
gap = output_gap(actual_gdp, potential_gdp)
print(f"需給ギャップ: {gap:.2f}%")
print(f"失業率（オークンの法則）: {okun_unemployment(gap, natural_rate):.2f}%")
print(f"実質利子率: {real_interest_rate(nominal_rate, inflation):.2f}%")
print(f"物価水準（貨幣数量説）: "
      f"{price_level_by_quantity_theory(nominal_money, velocity, real_income):.2f}")

In [ ]:
from lib.chapter14.plot import plot_phillips_curve

plot_phillips_curve(natural_rate, expected_inflation, phillips_slope)

## 試してみよう

- `government_after` を 150 にすると、古典派の総供給は垂直なので、所得は 1250 の
  まま、物価だけが 1.000 から 1.235 に上がります（財政政策は、物価を上げるだけ）
- `full_employment_income` を 1300 にすると、均衡の物価は 0.921 に下がり、所得は
  1300 になります
- `unemployment` を 6 にすると、物価上昇率は 1.00%、2 だと 3.00% です
  （失業率が低いほど、物価上昇率が高い）。`phillips_slope` を 1 にすると、2 のとき
  4.00% です
- `actual_gdp` を 1020 にすると、需給ギャップは 2.00%、失業率は 3.00% になります
- `nominal_money` を 1000 にすると、物価水準は 1.00 から 2.00 になります
  （貨幣供給が 2 倍で、物価も 2 倍）。`nominal_rate` が 5 で `inflation` が 6 なら、
  実質利子率は -1.00% です